# Zuo CIV batch fit with widths classified after fitting

Fit each of the 19 Zuo et al. spectra with two and three CIV Gaussians. Every CIV Gaussian has the same positive width floor and 35,250 km/s FWHM ceiling; no Gaussian is designated broad or narrow during fitting. After fitting, classify each CIV Gaussian at 1600 km/s FWHM. A model with no broad CIV Gaussian, a failed optimizer status, or non-finite BIC is a failed fit and contributes no measurements.

Use the matched notebook's 1445–1465 and 1700–1705 Å continuum windows, power-law continuum, and broad plus narrow He II and O III] terms in the 1500–1700 Å line complex. Apply the fixed Zuo rest-frame masks, PyQSOFit bad-pixel rejection, and continuum absorption rejection. PyQSOFit's iterative line rejection is disabled because it independently removes different pixels for each candidate model, including positive CIV peak residuals. Both models therefore fit the same line pixels; their exact fitted wavelengths are checked before model selection. Use the original inverse-variance errors without an extra S/N cut or downweighting.

Each CIV Gaussian has a distinct, data-scaled starting amplitude and initial width, with identical allowed bounds. MCMC uses 200 steps, 20 burn, and 10 thin; no Monte Carlo resampling. Select the three-CIV-component fit only when its BIC is at least 10 lower than the two-CIV-component fit. Compute legacy and three literature asymmetries, plus their uncertainties directly from the selected fit's MCMC samples, using only its continuous broad CIV profile.

Write tracking and CSV results to `Target_lists/unconstrained2/` and each fit's FITS/PDF products to `CIV_galaxy_spectra_targets/zuo_civ_data/unconstrained2/`. Restart the kernel and run all cells when ready. Older CSV and FITS/PDF files in those locations remain results of the previous procedure until the notebook is rerun.


In [ ]:
%matplotlib inline
from pathlib import Path
import importlib.util
import tempfile
import timeit
import warnings
import sys

import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.table import Table

PROJECT_ROOT = Path('/Users/nayera/PyQSOFit')
sys.path.insert(0, str(PROJECT_ROOT))
from pyqsofit.PyQSOFit import QSOFit
from data_calculations import blueshift_calc, asym_calc

warnings.filterwarnings('ignore')
QSOFit.set_mpl_style()


In [ ]:
# Inputs and new output locations
TARGET_CSV = PROJECT_ROOT / 'Target_lists' / 'zuo_targets_data.csv'
SPECTRA_ROOT = PROJECT_ROOT / 'CIV_galaxy_spectra_targets' / 'zuo_civ_data' / 'base_specs'
FIT_RESULTS_DIR = SPECTRA_ROOT.parent / 'unconstrained2'
TABLE_RESULTS_DIR = PROJECT_ROOT / 'Target_lists' / 'unconstrained2'
TRACKING_CSV = TABLE_RESULTS_DIR / 'zuo_civ_batch_fit_inputs.csv'
RESULTS_CSV = TABLE_RESULTS_DIR / 'zuo_civ_all_model_outputs.csv'
BEST_RESULTS_CSV = TABLE_RESULTS_DIR / 'zuo_civ_best_bic_outputs.csv'
HBETA_CSV = PROJECT_ROOT / 'Target_lists' / 'zuo_civ_hbeta_fwhm_ratio_comparison.csv'
FIGURE_READY_CSV = TABLE_RESULTS_DIR / 'zuo_civ_figure7_asymmetry_ready.csv'
ASYM_CALC_SCRIPT = Path('/Users/nayera/Astro-research/scripts/asym_calc.py')
WAVE_RANGE = np.array([1445.0, 1705.0])
CIV_LINE_RANGE = np.array([1500.0, 1700.0])

RUN_MCMC = True
NSAMP = 200
NBURN = 20
NTHIN = 10

# PyQSOFit uses sigma in natural-log wavelength. The small floor only keeps
# Gaussian widths positive; it is far below the broad/narrow boundary.
MIN_CIV_SIGMA = 1e-6
MAX_CIV_SIGMA = 0.04869268872068444  # 35,250 km/s FWHM
MAX_HEII_OIII_SIGMA = 0.01976249149738505  # matched notebook
NARROW_FWHM_LIMIT_KMS = 1600
NARROW_CIV_MAXSIG = float(np.float32(0.0022622972089434406))
BROAD_FWHM_THRESHOLD = np.nextafter(
    2 * np.sqrt(2 * np.log(2)) * (np.exp(NARROW_CIV_MAXSIG) - 1) * 300000.,
    np.inf,
)
BIC_IMPROVEMENT_FOR_THREE = 10.0
# These starts separate otherwise identical components; bounds remain identical.
CIV_INITIAL_SIGMAS = {2: (0.003, 0.012), 3: (0.002, 0.006, 0.018)}
CIV_INITIAL_SCALE_FRACTIONS = {2: (0.6, 0.3), 3: (0.5, 0.3, 0.2)}
# Four He II/O III] Gaussians have 12 parameters; two are tied.
NON_CIV_FREE_LINE_PARAMETERS = 10
CIV_BROAD_PROFILE_GRID_POINTS = 5000
CIV_BROAD_PROFILE_NSIGMA = 6.0

# Use the literature equations from the existing analysis script, while
# calculating posterior errors from complete MCMC draws in this notebook.
asym_spec = importlib.util.spec_from_file_location('authoritative_asym_calc', ASYM_CALC_SCRIPT)
literature_asymmetry = importlib.util.module_from_spec(asym_spec)
asym_spec.loader.exec_module(literature_asymmetry)

MODEL_GRID = {
    '2comp': {'civ_components': 2},
    '3comp': {'civ_components': 3},
}
MODEL_GRID_COMPONENT_COUNTS = {cfg['civ_components'] for cfg in MODEL_GRID.values()}

# Converted from the SDSS/FITS-redshift rest frame into the Zuo-paper
# rest frame: lambda_zuo = lambda_fits * (1 + z_fits) / (1 + z_zuo).
REST_FRAME_MASKS = {
    '0401': [(1535.361, 1538), (1539,1540.5), (1544.299, 1545.789), (1547.278, 1549.265)],
    '0725': [(1580.850, 1583.841), (1584.837, 1588.824)],
    '0731': [(1539.771, 1542.776), (1547.785, 1550.791)],
    '2608': [(1536.913, 1542.936)],
    '3683': [(1533, 1538)],
    '2825': [(1539.5, 1541.5)],
    '3699': [(1526.811, 1531.822)],
    '3804': [(1504.639, 1508.591), (1526,1527.5), (1546.1,1547.5), (1548.603, 1552.555)],
    '4206': [(1559.599, 1562.098), (1563.597, 1565.596), (1566.595, 1570.093)],
    '4604': [(1510.138, 1512.140), (1534.172, 1536.675), (1538, 1541.182), (1590.251, 1593.255)],
    '5496': [(1529.493, 1531.490)],
}

# Set to a small integer for a smoke test, then back to None for the full run.
TARGET_LIMIT = None


In [ ]:
def as_serializable(value):
    if value is None:
        return ''
    arr = np.asarray(value)
    return arr.tolist()


def spectrum_prefix(file_name):
    parts = Path(file_name).stem.split('-')
    if len(parts) >= 2 and parts[0] == 'spec':
        return parts[1]
    return parts[0]


def mask_for_spectrum(file_name):
    regions = REST_FRAME_MASKS.get(spectrum_prefix(file_name))
    return None if regions is None else np.asarray(regions, dtype=float)


REQUIRED_COLUMNS = {'name_sdss', 'z', 'ra', 'dec', 'plate', 'mjd', 'fiber', 'fits_path'}
SOURCE_COLUMNS = None


def inspect_spectrum(fits_path, z):
    if not fits_path.exists():
        return False, np.nan, np.nan, 'missing FITS file'
    try:
        with fits.open(fits_path) as hdul:
            names = set(hdul[1].columns.names)
            missing_arrays = sorted({'loglam', 'flux', 'ivar'} - names)
            if missing_arrays:
                return False, np.nan, np.nan, f'missing FITS arrays: {missing_arrays}'
            obs_wave = 10 ** np.asarray(hdul[1].data['loglam'], dtype=float)
        rest_min = float(np.nanmin(obs_wave) / (1 + z))
        rest_max = float(np.nanmax(obs_wave) / (1 + z))
        if rest_min > WAVE_RANGE[0] or rest_max < WAVE_RANGE[1]:
            reason = f'insufficient rest-frame coverage: {rest_min:.1f}--{rest_max:.1f} A'
            return False, rest_min, rest_max, reason
        return True, rest_min, rest_max, ''
    except Exception as exc:
        return False, np.nan, np.nan, f'cannot read FITS file: {exc!r}'


def load_targets(target_csv=TARGET_CSV):
    global SOURCE_COLUMNS
    df = pd.read_csv(target_csv)
    SOURCE_COLUMNS = list(df.columns)
    missing_columns = sorted(REQUIRED_COLUMNS - set(df.columns))
    if missing_columns:
        raise ValueError(f'Missing required columns in {target_csv.name}: {missing_columns}')

    df = df[df['name_sdss'].notna() & df['fits_path'].notna()].copy()
    df['target_name'] = df['name_sdss'].astype(str).str.strip()
    if df['target_name'].duplicated().any():
        duplicates = df.loc[df['target_name'].duplicated(False), 'target_name'].tolist()
        raise ValueError(f'Duplicate target names in {target_csv.name}: {duplicates}')
    df['file_name'] = df['fits_path'].apply(lambda value: Path(str(value)).name)
    # Resolve against the project folder so the notebook does not depend on CSV absolute paths.
    df['input_fits_path'] = df['fits_path'].astype(str)
    df['fits_path'] = df['file_name'].apply(lambda value: SPECTRA_ROOT / value)
    df['fit_z'] = pd.to_numeric(df['z'], errors='coerce')
    for column in ['ra', 'dec', 'plate', 'mjd', 'fiber']:
        df[column] = pd.to_numeric(df[column], errors='coerce')
    df['fit_z_source'] = 'Zuo2020 z'
    df['range_arr'] = [WAVE_RANGE.copy() for _ in range(len(df))]
    df['mask_arr'] = df['file_name'].apply(mask_for_spectrum)

    inspections = []
    for row in df.itertuples(index=False):
        reasons = []
        if not np.isfinite(row.fit_z) or row.fit_z <= 0:
            valid, rest_min, rest_max = False, np.nan, np.nan
            reasons.append('missing/non-positive fit redshift')
        else:
            valid, rest_min, rest_max, spectrum_reason = inspect_spectrum(
                Path(row.fits_path), float(row.fit_z)
            )
            if spectrum_reason:
                reasons.append(spectrum_reason)
        missing_metadata = [
            column for column in ['ra', 'dec', 'plate', 'mjd', 'fiber']
            if not np.isfinite(getattr(row, column))
        ]
        if missing_metadata:
            valid = False
            reasons.append(f'missing/non-numeric metadata: {missing_metadata}')
        inspections.append((valid, rest_min, rest_max, '; '.join(reasons)))
    df[['spectrum_valid', 'rest_wave_min', 'rest_wave_max', 'skip_reason']] = pd.DataFrame(
        inspections, index=df.index
    )
    df['run_fit'] = df['spectrum_valid'] & df['fit_z'].notna() & (df['fit_z'] > 0)

    export = df.copy()
    export['fits_path'] = export['fits_path'].astype(str)
    export['range_arr'] = export['range_arr'].apply(as_serializable)
    export['mask_arr'] = export['mask_arr'].apply(as_serializable)
    export.to_csv(TRACKING_CSV, index=False)
    return df


TABLE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
targets = load_targets()
fit_targets = targets[targets['run_fit']].copy()
if TARGET_LIMIT is not None:
    fit_targets = fit_targets.head(TARGET_LIMIT).copy()

print(f'Loaded {len(targets)} Zuo et al. targets from {TARGET_CSV.name}.')
print(f'{len(fit_targets)} targets passed the FITS/redshift/coverage preflight.')
print(f'Tracking table written to {TRACKING_CSV}.')
if (~targets['run_fit']).any():
    display(targets.loc[~targets['run_fit'], ['target_name', 'file_name', 'fit_z', 'skip_reason']])
    raise RuntimeError('Preflight failed; fix the listed inputs before starting the batch fits.')

display(fit_targets[['target_name', 'file_name', 'fit_z', 'range_arr', 'mask_arr', 'rest_wave_min', 'rest_wave_max', 'fits_path']])


In [ ]:
# Match zuo_matched's fixed continuum windows.
CONTI_WINDOWS = np.rec.array(
    [(1445.0, 1465.0), (1700.0, 1705.0)],
    formats='float32,float32', names='min,max',
)


def estimate_civ_initial_height(lam, flux, err, z, mask_arr):
    """Get a robust CIV peak estimate for starting amplitudes, not for masking."""
    rest_wave = np.asarray(lam, dtype=float) / (1 + z)
    valid = np.isfinite(rest_wave) & np.isfinite(flux) & np.isfinite(err) & (err > 0) & (flux != 0)
    if mask_arr is not None:
        for lo, hi in mask_arr:
            valid &= ~((rest_wave > lo) & (rest_wave < hi))
    continuum = valid & (((rest_wave > 1445) & (rest_wave < 1465)) |
                         ((rest_wave > 1700) & (rest_wave < 1705)))
    core = valid & (rest_wave > 1510) & (rest_wave < 1580)
    if continuum.sum() < 5 or core.sum() < 5:
        raise RuntimeError('Insufficient usable continuum or CIV pixels for initial amplitudes')
    observed_height = np.percentile(flux[core], 95) - np.median(flux[continuum])
    return max(float(observed_height * (1 + z)), 1.0)


def make_qsopar(param_dir, civ_components, civ_initial_height):
    primary_hdu = fits.PrimaryHDU(header=fits.Header({'Author': 'Nayera Abdessalam'}))
    # Separate one-Gaussian rows let PyQSOFit use distinct initial values.
    # Every CIV row retains exactly the same allowed sigma/center/scale bounds.
    civ_rows = [
        (1549.06, 'CIV', 1500, 1700, f'CIV_g{i}', 1,
         civ_initial_height * fraction, 0.0, 1e10,
         sigma, MIN_CIV_SIGMA, MAX_CIV_SIGMA, 0.015, 0, 0, 0, 0.05, 1)
        for i, (fraction, sigma) in enumerate(zip(
            CIV_INITIAL_SCALE_FRACTIONS[civ_components], CIV_INITIAL_SIGMAS[civ_components]
        ), start=1)
    ]
    other_rows = [
        (1640.42, 'CIV', 1500, 1700, 'HeII_br', 1, 0.0, 0.0, 1e10,
         0.005, 0.004, MAX_HEII_OIII_SIGMA, 0.008, 0, 0, 0, 0.05, 1),
        (1640.42, 'CIV', 1500, 1700, 'HeII_na', 1, 0.0, 0.0, 1e10,
         1e-3, 5e-4, NARROW_CIV_MAXSIG, 0.01, 1, 1, 0, 0.002, 1),
        (1663.48, 'CIV', 1500, 1700, 'OIII1663_br', 1, 0.0, 0.0, 1e10,
         0.005, 0.004, MAX_HEII_OIII_SIGMA, 0.008, 0, 0, 0, 0.05, 1),
        (1663.48, 'CIV', 1500, 1700, 'OIII1663_na', 1, 0.0, 0.0, 1e10,
         1e-3, 5e-4, NARROW_CIV_MAXSIG, 0.01, 1, 1, 0, 0.002, 1),
    ]
    line_priors = np.rec.array(
        civ_rows + other_rows,
        formats='float32,a20,float32,float32,a20,int32,float32,float32,float32,float32,float32,float32,float32,int32,int32,int32,float32,int32',
        names='lambda,compname,minwav,maxwav,linename,ngauss,inisca,minsca,maxsca,inisig,minsig,maxsig,voff,vindex,windex,findex,fvalue,vary',
    )
    hdu1 = fits.BinTableHDU(data=line_priors, name='line_priors')
    hdu2 = fits.BinTableHDU(data=CONTI_WINDOWS, name='conti_windows')

    conti_priors = np.rec.array([
        ('Fe_uv_norm', 0.0, 0.0, 1e10, 1),
        ('Fe_uv_FWHM', 3000, 1200, 18000, 1),
        ('Fe_uv_shift', 0.0, -0.01, 0.01, 1),
        ('Fe_op_norm', 0.0, 0.0, 1e10, 1),
        ('Fe_op_FWHM', 3000, 1200, 18000, 1),
        ('Fe_op_shift', 0.0, -0.01, 0.01, 1),
        ('PL_norm', 1.0, 0.0, 1e10, 1),
        ('PL_slope', -1.5, -5.0, 3.0, 1),
        ('Blamer_norm', 0.0, 0.0, 1e10, 1),
        ('Balmer_Te', 15000, 10000, 50000, 1),
        ('Balmer_Tau', 0.5, 0.1, 2.0, 1),
        ('conti_a_0', 0.0, None, None, 1),
        ('conti_a_1', 0.0, None, None, 1),
        ('conti_a_2', 0.0, None, None, 1),
    ], formats='a20,float32,float32,float32,int32', names='parname,initial,min,max,vary')
    hdu3 = fits.BinTableHDU(data=conti_priors, name='conti_priors')
    measure_info = Table([[[1350, 1450, 3000, 4200, 5100]], [[[1500, 1550]]]],
                         names=['cont_loc', 'Fe_flux_range'], dtype=['float32', 'float32'])
    hdu4 = fits.BinTableHDU(data=measure_info, name='measure_info')
    param_path = Path(param_dir) / 'qsopar.fits'
    fits.HDUList([primary_hdu, hdu1, hdu2, hdu3, hdu4]).writeto(param_path, overwrite=True)
    return param_path


In [ ]:
def read_sdss_spectrum(fits_path):
    with fits.open(fits_path) as data:
        lam = 10 ** data[1].data['loglam']
        flux = np.asarray(data[1].data['flux'], dtype=float)
        ivar = np.asarray(data[1].data['ivar'], dtype=float)
        err = np.full_like(flux, np.inf, dtype=float)
        good_ivar = np.isfinite(ivar) & (ivar > 0)
        err[good_ivar] = 1 / np.sqrt(ivar[good_ivar])
        header = data[0].header
        meta = {
            'ra': header.get('plug_ra', np.nan),
            'dec': header.get('plug_dec', np.nan),
            'plateid': header.get('plateid', -1),
            'mjd': header.get('mjd', -1),
            'fiberid': header.get('fiberid', -1),
        }
    usable = np.isfinite(flux) & np.isfinite(err) & (err > 0) & (flux != 0)
    meta['usable_pixels'] = int(np.sum(usable))
    meta['unusable_pixels'] = int(np.sum(~usable))
    meta['total_pixels'] = int(len(flux))
    return lam, flux, err, meta


def line_fit_wavelengths(q):
    """Return exactly the pixels PyQSOFit fits when rej_abs_line is disabled."""
    wave = np.asarray(q.wave, dtype=float)
    # PyQSOFit.fit_lines always removes points more than 1 sigma below its
    # continuum in this complex; the remaining rule is model-independent.
    selected = (wave > CIV_LINE_RANGE[0]) & (wave < CIV_LINE_RANGE[1])
    selected &= ~(np.asarray(q.line_flux) < -np.asarray(q.err))
    return wave[selected]


def named_pyqsofit_results(q):
    """Return every scalar continuum and line result with collision-safe names."""
    output = {}
    for prefix, names, values in (
        ('pyqsofit_conti', q.conti_result_name, q.conti_result),
        ('pyqsofit_line', q.line_result_name, q.line_result),
    ):
        for name, value in zip(names, values):
            scalar = np.ravel(value)[0]
            output[f'{prefix}_{name}'] = scalar.item() if hasattr(scalar, 'item') else scalar
    return output


def source_metadata(target_row):
    """Carry the complete Zuo input row into each result for combined analysis."""
    return {
        f'zuo_{column}': (target_row['input_fits_path'] if column == 'fits_path' else target_row[column])
        for column in SOURCE_COLUMNS
    }


def get_line_value(q, key, default=np.nan):
    idx = np.where(np.asarray(q.line_result_name) == key)[0]
    if idx.size == 0:
        return default
    return float(np.ravel(q.line_result[idx[0]])[0])


def civ_component_values(q, line_type, sample_index=None):
    """Measure all CIV rows together, classified by fitted sigma."""
    params = all_civ_params(q, sample_index=sample_index)
    return np.asarray(q.line_prop(1549.06, params.reshape(-1), line_type,
                                  ln_sigma_br=NARROW_CIV_MAXSIG), dtype=float)


def component_props(q, line_type):
    names = ['FWHM', 'sigma', 'EW', 'peak', 'area', 'snr']
    values = civ_component_values(q, line_type)
    out = dict(zip(names, values))
    if getattr(q, 'MCMC', False) and len(getattr(q, 'gauss_result_all', [])) > 0:
        samples = np.asarray([
            civ_component_values(q, line_type, sample_index=i)
            for i in range(q.gauss_result_all.shape[0])
        ], dtype=float)
        for j, name in enumerate(names):
            out[f'{name}_err_mcmc'] = posterior_error(samples[:, j])
    else:
        for name in names:
            out[f'{name}_err_mcmc'] = np.nan
    return out


def gaussian_params(q, line_name, sample_index=None):
    result_names = np.asarray(q.gauss_result_name)
    result = np.asarray(q.gauss_result, dtype=object)
    params = []
    i = 1
    while True:
        keys = [f'{line_name}_{i}_scale', f'{line_name}_{i}_centerwave', f'{line_name}_{i}_sigma']
        if not all(key in result_names for key in keys):
            break
        if sample_index is None:
            vals = [float(np.ravel(result[result_names == key][0])[0]) for key in keys]
        else:
            # gauss_result_name interleaves value/error, while gauss_result_all stores values only.
            indices = [int(np.flatnonzero(result_names == key)[0] // 2) for key in keys]
            vals = [float(q.gauss_result_all[sample_index, index]) for index in indices]
        params.append(vals)
        i += 1
    return np.asarray(params, dtype=float)


def all_civ_params(q, sample_index=None):
    names = [str(name) for name in q.linelist['linename'] if str(name).startswith('CIV_g')]
    if len(names) not in MODEL_GRID_COMPONENT_COUNTS:
        raise RuntimeError(f'Expected two or three fitted CIV rows, found {len(names)}')
    parts = [gaussian_params(q, name, sample_index=sample_index) for name in names]
    if any(part.shape != (1, 3) for part in parts):
        raise RuntimeError('Missing fitted CIV Gaussian parameters')
    return np.vstack(parts)


def classified_civ_params(q, line_type, sample_index=None):
    params = all_civ_params(q, sample_index=sample_index)
    if params.size == 0:
        return np.empty((0, 3), dtype=float)
    if line_type == 'broad':
        selected = params[:, 2] > NARROW_CIV_MAXSIG
    elif line_type == 'narrow':
        selected = (params[:, 2] > 0) & (params[:, 2] <= NARROW_CIV_MAXSIG)
    else:
        raise ValueError(f'Unknown CIV line type: {line_type}')
    return params[selected]


def broad_civ_profile(q, sample_index=None):
    """Evaluate only fitted broad CIV Gaussians on a continuous model grid."""
    params = classified_civ_params(q, 'broad', sample_index=sample_index)
    if params.size == 0 or not np.isfinite(params).all() or np.any(params[:, 2] <= 0):
        return None, None
    lnlam = np.linspace(
        np.min(params[:, 1] - CIV_BROAD_PROFILE_NSIGMA * params[:, 2]),
        np.max(params[:, 1] + CIV_BROAD_PROFILE_NSIGMA * params[:, 2]),
        CIV_BROAD_PROFILE_GRID_POINTS,
    )
    lam = np.exp(lnlam)
    model = np.zeros_like(lam, dtype=float)
    for amp, center, sigma in params:
        model += amp * np.exp(-0.5 * ((lnlam - center) / sigma) ** 2)
    if not np.isfinite(model).all() or np.max(model) <= 0:
        return None, None
    return lam, model


def broad_half_wavelengths(q, sample_index=None):
    lam, model = broad_civ_profile(q, sample_index=sample_index)
    if lam is None:
        return np.nan, np.nan, np.nan
    i_peak = int(np.argmax(model))
    half_peak = 0.5 * model[i_peak]
    left_below = np.flatnonzero(model[:i_peak] < half_peak)
    right_below = np.flatnonzero(model[i_peak + 1:] < half_peak)
    if left_below.size == 0 or right_below.size == 0:
        return np.nan, np.nan, np.nan

    def crossing(j0, j1):
        y0, y1 = model[j0], model[j1]
        return float(lam[j0] + (half_peak - y0) * (lam[j1] - lam[j0]) / (y1 - y0))

    left0 = int(left_below[-1])
    right1 = int(i_peak + 1 + right_below[0])
    lam_blue = crossing(left0, left0 + 1)
    lam_red = crossing(right1 - 1, right1)
    increments = 0.5 * (model[1:] + model[:-1]) * np.diff(lam)
    cumulative = np.concatenate(([0.0], np.cumsum(increments)))
    if cumulative[-1] <= 0:
        return lam_blue, lam_red, np.nan
    lam_half = float(np.interp(0.5 * cumulative[-1], cumulative, lam))
    return lam_blue, lam_red, lam_half


BROAD_ASYMMETRY_INPUT_KEYS = (
    'lam_p05', 'lam_p50', 'lam_p95',
    'lam_blue_15pct', 'lam_red_15pct',
    'lam_blue_25pct', 'lam_red_25pct',
    'lam_blue_80pct', 'lam_red_80pct',
    'lam_peak',
)


def broad_asymmetry_input_wavelengths(q, sample_index=None):
    """Measure shape wavelengths from the continuous broad-C IV-only model."""
    empty = {key: np.nan for key in BROAD_ASYMMETRY_INPUT_KEYS}
    lam, model = broad_civ_profile(q, sample_index=sample_index)
    if lam is None:
        return empty

    i_peak = int(np.nanargmax(model))
    peak_flux = float(model[i_peak])

    def interpolate_crossing(j0, j1, target):
        y0, y1 = model[j0], model[j1]
        if y1 == y0:
            return np.nan
        return float(lam[j0] + (target - y0) * (lam[j1] - lam[j0]) / (y1 - y0))

    def crossings(fraction):
        target = fraction * peak_flux
        left_below = np.where(model[:i_peak] < target)[0]
        right_below = np.where(model[i_peak + 1:] < target)[0]
        if left_below.size == 0 or right_below.size == 0:
            return np.nan, np.nan
        left0 = int(left_below[-1])
        right1 = int(i_peak + 1 + right_below[0])
        return (
            interpolate_crossing(left0, left0 + 1, target),
            interpolate_crossing(right1 - 1, right1, target),
        )

    lam_blue_15pct, lam_red_15pct = crossings(0.15)
    lam_blue_25pct, lam_red_25pct = crossings(0.25)
    lam_blue_80pct, lam_red_80pct = crossings(0.80)

    increments = 0.5 * (model[1:] + model[:-1]) * np.diff(lam)
    cumulative = np.concatenate(([0.0], np.cumsum(increments)))
    if cumulative[-1] <= 0:
        return empty
    lam_p05, lam_p50, lam_p95 = np.interp(
        np.array([0.05, 0.50, 0.95]) * cumulative[-1], cumulative, lam
    )

    return {
        'lam_p05': float(lam_p05),
        'lam_p50': float(lam_p50),
        'lam_p95': float(lam_p95),
        'lam_blue_15pct': lam_blue_15pct,
        'lam_red_15pct': lam_red_15pct,
        'lam_blue_25pct': lam_blue_25pct,
        'lam_red_25pct': lam_red_25pct,
        'lam_blue_80pct': lam_blue_80pct,
        'lam_red_80pct': lam_red_80pct,
        'lam_peak': float(lam[i_peak]),
    }


def posterior_error(values):
    finite = np.asarray(values, dtype=float)
    finite = finite[np.isfinite(finite)]
    if finite.size == 0:
        return np.nan
    lo, hi = np.nanpercentile(finite, [16, 84])
    return float((hi - lo) / 2)


ASYMMETRY_COLUMNS = (
    'CIV_asym', 'CIV_asym_err_mcmc',
    'CIV_asym_zg_R', 'CIV_asym_zg_R_err_mcmc',
    'CIV_asym_zg_R_norm_0_2', 'CIV_asym_zg_R_norm_0_2_err_mcmc',
    'CIV_asym_wolf_15_80', 'CIV_asym_wolf_15_80_err_mcmc',
    'CIV_asym_wolf_15_80_norm_0_2', 'CIV_asym_wolf_15_80_norm_0_2_err_mcmc',
    'CIV_asym_4de1_AI_1_4', 'CIV_asym_4de1_AI_1_4_err_mcmc',
    'CIV_asym_4de1_AI_1_4_norm_0_2', 'CIV_asym_4de1_AI_1_4_norm_0_2_err_mcmc',
)


def asymmetry_values(lam_blue, lam_red, peak, shape):
    """Use the existing literature equations for scalars or posterior arrays."""
    zg = literature_asymmetry.zakamska_greene_r(
        shape['lam_p05'], shape['lam_p50'], shape['lam_p95']
    )
    wolf = literature_asymmetry.wolf_asymmetry(
        shape['lam_blue_15pct'], shape['lam_red_15pct'],
        shape['lam_blue_80pct'], shape['lam_red_80pct'],
    )
    ai = literature_asymmetry.ai_quarter(
        shape['lam_blue_25pct'], shape['lam_red_25pct'], shape['lam_peak']
    )
    return {
        'CIV_asym': literature_asymmetry.asym_calc(lam_blue, lam_red, peak),
        'CIV_asym_zg_R': zg,
        'CIV_asym_zg_R_norm_0_2': literature_asymmetry.scale_bounded_asymmetry(zg, blue_sign=-1),
        'CIV_asym_wolf_15_80': wolf,
        'CIV_asym_wolf_15_80_norm_0_2': literature_asymmetry.scale_bounded_asymmetry(wolf, blue_sign=1),
        'CIV_asym_4de1_AI_1_4': ai,
        'CIV_asym_4de1_AI_1_4_norm_0_2': literature_asymmetry.scale_bounded_asymmetry(ai, blue_sign=-1),
    }


def broad_profile_mcmc_results(q):
    """Propagate whole MCMC draws through classification and profile equations."""
    samples = getattr(q, 'gauss_result_all', None)
    if not getattr(q, 'MCMC', False) or samples is None or len(samples) == 0:
        raise RuntimeError('Selected model has no CIV MCMC samples')
    n_samples = samples.shape[0]
    profiles = [broad_half_wavelengths(q, sample_index=i) for i in range(n_samples)]
    lam_blue = np.asarray([profile[0] for profile in profiles], dtype=float)
    lam_red = np.asarray([profile[1] for profile in profiles], dtype=float)
    lam_half = np.asarray([profile[2] for profile in profiles], dtype=float)
    shapes = [broad_asymmetry_input_wavelengths(q, sample_index=i) for i in range(n_samples)]
    shape_samples = {
        key: np.asarray([values[key] for values in shapes], dtype=float)
        for key in BROAD_ASYMMETRY_INPUT_KEYS
    }
    broad_properties = np.asarray([
        civ_component_values(q, 'broad', sample_index=i)
        for i in range(n_samples)
    ], dtype=float)
    peaks = broad_properties[:, 3]
    asymmetries = asymmetry_values(lam_blue, lam_red, peaks, shape_samples)
    blueshifts = blueshift_calc(lam_half)
    result = {
        'CIV_blueshift_err_mcmc': posterior_error(blueshifts),
        'lam_blue_err_mcmc': posterior_error(lam_blue),
        'lam_red_err_mcmc': posterior_error(lam_red),
        'lam_half_err_mcmc': posterior_error(lam_half),
        'mcmc_samples_returned': int(n_samples),
        'mcmc_valid_blueshift_samples': int(np.isfinite(blueshifts).sum()),
    }
    for key, values in shape_samples.items():
        result[f'{key}_err_mcmc'] = posterior_error(values)
    for key, values in asymmetries.items():
        result[f'{key}_err_mcmc'] = posterior_error(values)
        result[f'mcmc_valid_{key}_samples'] = int(np.isfinite(values).sum())
    return result


In [ ]:
def fit_target_model(target_row, model_name, model_cfg):
    target = target_row['target_name']
    fits_path = Path(target_row['fits_path'])
    z = float(target_row['fit_z'])
    range_arr = np.asarray(target_row['range_arr'], dtype=float)
    mask_arr = target_row['mask_arr']
    if mask_arr is not None:
        mask_arr = np.asarray(mask_arr, dtype=float)
    output_stem = f'{target}_{model_name}'
    output_fits_path = FIT_RESULTS_DIR / f'{output_stem}.fits'
    output_pdf_path = FIT_RESULTS_DIR / f'{output_stem}.pdf'
    lam, flux, err, meta = read_sdss_spectrum(fits_path)
    civ_initial_height = estimate_civ_initial_height(lam, flux, err, z, mask_arr)

    with tempfile.TemporaryDirectory(prefix='pyqsofit_civ_') as param_dir:
        make_qsopar(param_dir, model_cfg['civ_components'], civ_initial_height)
        q = QSOFit(lam, flux, err, z,
                   ra=meta['ra'], dec=meta['dec'], plateid=meta['plateid'],
                   mjd=meta['mjd'], fiberid=meta['fiberid'], path=param_dir)
        start = timeit.default_timer()
        q.Fit(name=output_stem,
              nsmooth=1,
              and_mask=False,
              or_mask=False,
              reject_badpix=True,
              deredden=False,
              wave_range=range_arr,
              wave_mask=mask_arr,
              decompose_host=False,
              host_prior=False,
              host_prior_scale=0.2,
              host_line_mask=True,
              decomp_na_mask=True,
              qso_type='CZBIN1',
              npca_qso=10,
              host_type='PCA',
              npca_gal=5,
              Fe_uv_op=False,
              poly=False,
              BC=False,
              initial_guess=None,
              rej_abs_conti=True,
              n_pix_min_conti=30,
              linefit=True,
              rej_abs_line=False,  # one fixed set of CIV pixels for both models
              MC=False,
              MCMC=RUN_MCMC,
              nsamp=NSAMP,
              param_file_name='qsopar.fits',
              nburn=NBURN,
              nthin=NTHIN,
              epsilon_jitter=0.,  # distinct CIV starts are set in qsopar.fits
              save_result=True,
              save_fits_path=str(FIT_RESULTS_DIR),
              save_fits_name=output_stem,
              plot_fig=True,
              save_fig=True,
              plot_corner=False,
              verbose=False,
              kwargs_plot={'broad_fwhm': BROAD_FWHM_THRESHOLD,
                           'save_fig_path': str(FIT_RESULTS_DIR)},
              kwargs_conti_emcee={},
              kwargs_line_emcee={})
        fit_seconds = float(timeit.default_timer() - start)

    # Classify the fitted CIV Gaussians only after PyQSOFit has finished.
    broad = classified_civ_params(q, 'broad')
    if broad.size == 0:
        raise RuntimeError(f'{target} {model_name}: no broad CIV Gaussian above 1600 km/s')
    narrow = classified_civ_params(q, 'narrow')
    bic = get_line_value(q, '1_line_bic')
    if not np.isfinite(bic):
        raise RuntimeError(f'{target} {model_name}: non-finite line-complex BIC')
    line_status = get_line_value(q, '1_line_status')
    if line_status != 1:
        raise RuntimeError(f'{target} {model_name}: line optimizer did not converge (status={line_status})')
    line_ndof = get_line_value(q, '1_ndof')
    line_fit_pixels = len(line_fit_wavelengths(q))
    expected_pixels = int(round(
        line_ndof + 3 * model_cfg['civ_components'] + NON_CIV_FREE_LINE_PARAMETERS
    ))
    if line_fit_pixels != expected_pixels:
        raise RuntimeError(f'{target} {model_name}: line-pixel count does not match PyQSOFit degrees of freedom')
    br = civ_component_values(q, 'broad')
    na = civ_component_values(q, 'narrow')
    row = {
        'Target': target,
        'Model': model_name,
        'file_name': target_row['file_name'],
        'fits_path': str(fits_path),
        'fit_z': z,
        'fit_z_source': target_row['fit_z_source'],
        'range_arr': as_serializable(range_arr),
        'mask_arr': as_serializable(mask_arr),
        'output_fits_path': str(output_fits_path),
        'output_pdf_path': str(output_pdf_path),
        'civ_components_fit': model_cfg['civ_components'],
        'civ_components_broad': len(broad),
        'civ_components_narrow': len(narrow),
        'narrow_fwhm_limit_kms': NARROW_FWHM_LIMIT_KMS,
        'reduced Chi squared': get_line_value(q, '1_line_red_chi2'),
        'BIC_mcmc': bic,
        'FWHM_CIV': float(br[0]),
        'na_FWHM': float(na[0]),
        'line_min_chi2': get_line_value(q, '1_line_min_chi2'),
        'line_status': line_status,
        'line_ndof': line_ndof,
        'line_fit_pixels': line_fit_pixels,
        'line_niter': get_line_value(q, '1_niter'),
        'usable_pixels': meta['usable_pixels'],
        'unusable_pixels': meta['unusable_pixels'],
        'total_pixels': meta['total_pixels'],
        'fit_seconds': fit_seconds,
        'status': 'ok',
        'error': '',
    }
    row.update(source_metadata(target_row))
    row.update(named_pyqsofit_results(q))
    return row, q


def selected_profile_measurements(q):
    """Calculate shape, asymmetries, and posterior errors for the selected fit."""
    br = component_props(q, 'broad')
    na = component_props(q, 'narrow')
    lam_blue, lam_red, lam_half = broad_half_wavelengths(q)
    shape = broad_asymmetry_input_wavelengths(q)
    central = asymmetry_values(lam_blue, lam_red, br['peak'], shape)
    result = {
        'lam_blue': lam_blue,
        'lam_red': lam_red,
        'lam_half': lam_half,
        'br_peak': br['peak'],
        'br_peak_err_mcmc': br['peak_err_mcmc'],
        'CIV_blueshift_kms': blueshift_calc(lam_half),
        'FWHM_err_mcmc': br['FWHM_err_mcmc'],
    }
    for prefix, props in (('br', br), ('na', na)):
        for key, value in props.items():
            result[f'{prefix}_{key}'] = value
    for key, value in shape.items():
        result[f'CIV_{key}_A'] = value
    for key, value in central.items():
        result[key] = float(value)
    result['CIV_asym'] = asym_calc(lam_blue, lam_red, br['peak'])
    posterior = broad_profile_mcmc_results(q)
    for key, value in posterior.items():
        if key in [f'{name}_err_mcmc' for name in BROAD_ASYMMETRY_INPUT_KEYS]:
            name = key.removesuffix('_err_mcmc')
            result[f'CIV_{name}_err_mcmc_A'] = value
        else:
            result[key] = value
    return result


In [ ]:
FIT_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
TABLE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
results = []
best_rows = []
expected = len(fit_targets) * len(MODEL_GRID)
print(f'Running {expected} fits: {len(fit_targets)} targets x 2 CIV models.')

for _, target_row in fit_targets.iterrows():
    target = target_row['target_name']
    fitted = {}
    target_rows = {}
    for model_name, model_cfg in MODEL_GRID.items():
        label = f'{target} model {model_name}'
        print(f'Starting {label}')
        try:
            row, q = fit_target_model(target_row, model_name, model_cfg)
            fitted[model_name] = q
            print(f"Finished {label}: BIC={row['BIC_mcmc']:.3g}, broad={row['civ_components_broad']}, narrow={row['civ_components_narrow']}")
        except Exception as exc:
            row = {
                'Target': target,
                'Model': model_name,
                'file_name': target_row['file_name'],
                'fits_path': str(target_row['fits_path']),
                'fit_z': target_row['fit_z'],
                'fit_z_source': target_row['fit_z_source'],
                'range_arr': as_serializable(target_row['range_arr']),
                'mask_arr': as_serializable(target_row['mask_arr']),
                'output_fits_path': str(FIT_RESULTS_DIR / f'{target}_{model_name}.fits'),
                'output_pdf_path': str(FIT_RESULTS_DIR / f'{target}_{model_name}.pdf'),
                'civ_components_fit': model_cfg['civ_components'],
                'narrow_fwhm_limit_kms': NARROW_FWHM_LIMIT_KMS,
                'status': 'failed',
                'error': repr(exc),
            }
            row.update(source_metadata(target_row))
            print(f'FAILED {label}: {exc!r}')
        results.append(row)
        target_rows[model_name] = row
        pd.DataFrame(results).to_csv(RESULTS_CSV, index=False)

    # Both BIC values are required for the specified 10-point comparison.
    if len(fitted) != 2:
        print(f'SKIPPED selection for {target}: one or both models failed.')
        continue
    pixels_two = line_fit_wavelengths(fitted['2comp'])
    pixels_three = line_fit_wavelengths(fitted['3comp'])
    if not np.array_equal(pixels_two, pixels_three):
        for row in target_rows.values():
            row['status'] = 'failed'
            row['error'] = f'line pixel mismatch: 2comp={len(pixels_two)}, 3comp={len(pixels_three)}'
        print(f'FAILED {target}: line pixel mismatch ({len(pixels_two)} versus {len(pixels_three)})')
        continue
    bic_two = target_rows['2comp']['BIC_mcmc']
    bic_three = target_rows['3comp']['BIC_mcmc']
    selected_name = '3comp' if bic_three <= bic_two - BIC_IMPROVEMENT_FOR_THREE else '2comp'
    selected_row = target_rows[selected_name]
    try:
        selected_measurements = selected_profile_measurements(fitted[selected_name])
        best_row = dict(selected_row)
        best_row.update(selected_measurements)
        best_row['BIC_2comp'] = bic_two
        best_row['BIC_3comp'] = bic_three
        best_row['delta_BIC_3_minus_2'] = bic_three - bic_two
        best_row['selection_rule'] = '3comp only if BIC_3comp <= BIC_2comp - 10'
        best_rows.append(best_row)
        selected_row['is_selected'] = True
        print(f'Selected {target} {selected_name}: delta BIC (3 minus 2)={bic_three - bic_two:.3g}')
    except Exception as exc:
        selected_row['status'] = 'failed'
        selected_row['error'] = f'selected profile measurements: {exc!r}'
        print(f'FAILED selected profile for {target}: {exc!r}')
    finally:
        fitted.clear()

results_df = pd.DataFrame(results)
if 'is_selected' not in results_df:
    results_df['is_selected'] = False
else:
    results_df['is_selected'] = results_df['is_selected'].fillna(False).astype(bool)
results_df['BIC_mcmc'] = pd.to_numeric(results_df.get('BIC_mcmc'), errors='coerce')
results_df['delta_BIC_3_minus_2'] = np.nan
for target, group in results_df.groupby('Target', sort=False):
    bics = group.set_index('Model')['BIC_mcmc']
    if {'2comp', '3comp'}.issubset(bics.index) and np.isfinite(bics['2comp']) and np.isfinite(bics['3comp']):
        results_df.loc[group.index, 'delta_BIC_3_minus_2'] = bics['3comp'] - bics['2comp']
results_df.to_csv(RESULTS_CSV, index=False)

if TARGET_LIMIT is None:
    assert len(fit_targets) == 19 and fit_targets['target_name'].nunique() == 19
assert len(results_df) == expected
assert not results_df.duplicated(['Target', 'Model']).any()
failed = results_df.loc[results_df['status'].ne('ok') | ~np.isfinite(results_df['BIC_mcmc']),
                        ['Target', 'Model', 'error']]
if not failed.empty or len(best_rows) != len(fit_targets):
    display(failed)
    raise RuntimeError(f'Only {len(best_rows)} of {len(fit_targets)} targets have two valid fits and selected measurements; see {RESULTS_CSV}')

best_results_df = pd.DataFrame(best_rows)
assert len(best_results_df) == len(fit_targets)
assert best_results_df['Target'].nunique() == len(fit_targets)
assert results_df['is_selected'].sum() == len(fit_targets)
assert set(best_results_df['Target']) == set(fit_targets['target_name'])

raw_profile_columns = [
    column
    for key in BROAD_ASYMMETRY_INPUT_KEYS
    for column in (f'CIV_{key}_A', f'CIV_{key}_err_mcmc_A')
]
required_finite = [
    'FWHM_CIV', 'FWHM_err_mcmc', 'CIV_blueshift_kms', 'CIV_blueshift_err_mcmc',
    *ASYMMETRY_COLUMNS, *raw_profile_columns,
]
if not np.isfinite(best_results_df[required_finite].to_numpy(dtype=float)).all():
    raise RuntimeError('Selected results contain non-finite CIV measurements or posterior errors')
if (best_results_df['FWHM_CIV'] <= 0).any():
    raise RuntimeError('A selected fit has non-positive broad CIV FWHM')

hbeta = pd.read_csv(HBETA_CSV)
hbeta_merge = hbeta[['name_sdss', 'zuo_hbeta_fwhm_km_s', 'zuo_hbeta_fwhm_err_km_s']].rename(columns={
    'name_sdss': 'Target',
    'zuo_hbeta_fwhm_km_s': 'fwhm_hb',
    'zuo_hbeta_fwhm_err_km_s': 'fwhm_hb_err',
})
figure_ready_df = best_results_df.merge(
    hbeta_merge, on='Target', how='left', validate='one_to_one', indicator=True
)
figure_ready_df.insert(2, 'sample', 'Zuo unconstrained')
if not figure_ready_df['_merge'].eq('both').all():
    raise RuntimeError('H-beta comparison table is missing one or more selected targets')
figure_columns = [
    'Target', 'Model', 'sample', 'FWHM_CIV', 'FWHM_err_mcmc',
    'fwhm_hb', 'fwhm_hb_err', *ASYMMETRY_COLUMNS, *raw_profile_columns,
    'BIC_mcmc', 'BIC_2comp', 'BIC_3comp', 'delta_BIC_3_minus_2', 'status',
]
figure_ready_df = figure_ready_df[figure_columns].copy()
best_results_df.to_csv(BEST_RESULTS_CSV, index=False)
figure_ready_df.to_csv(FIGURE_READY_CSV, index=False)
print(f'Validated {len(results_df)} fit rows and {len(best_results_df)} selected targets.')
display(best_results_df[['Target', 'Model', 'BIC_2comp', 'BIC_3comp', 'delta_BIC_3_minus_2']])
print(f'All-model output: {RESULTS_CSV}')
print(f'Selected-model output: {BEST_RESULTS_CSV}')
print(f'Figure-ready output: {FIGURE_READY_CSV}')


## Broad-profile measurements and selection

Every fitted CIV Gaussian has the same width bounds and a distinct starting amplitude and width. After fitting, individual CIV Gaussians with FWHM greater than 1600 km/s form the broad profile; narrower ones form the narrow profile. He II and O III] are fitted in the line complex but never enter the CIV profile measurements. A fit with no broad CIV Gaussian fails and its measurement fields are not recorded.

For both central values and MCMC draws, the broad-only CIV Gaussians are summed on a 5000-point log-wavelength grid extending six sigmas beyond the components. The notebook records 5%, 50%, and 95% cumulative-flux wavelengths, 15%, 25%, and 80% height crossings, the peak, the legacy asymmetry, Zakamska–Greene R, Wolf core-to-base asymmetry, and AI(1/4). The literature statistics also retain the fixed 0–2 transformations used in `Astro-research/scripts/asym_calc.py`. Each uncertainty is half the 84th minus 16th posterior percentile after calculating the statistic for each MCMC draw; this keeps the wavelength correlations and reclassifies CIV widths for each draw.

The three-component model is selected only when its BIC is at least 10 lower than the two-component BIC. Both fits must converge and use identical CIV line wavelengths for the comparison. Fixed Zuo masks, bad-pixel rejection, continuum absorption rejection, and PyQSOFit's model-independent below-continuum line mask remain active. Iterative model-dependent line rejection is disabled because it can reject the emission peak after a poor initial fit and give each model a different BIC sample size. The all-model table contains fit diagnostics and component counts; asymmetries and their posterior errors appear only in the selected-model and figure-ready tables. Tracking and CSV tables are written under `Target_lists/unconstrained2/`, while per-fit FITS/PDF files go under `CIV_galaxy_spectra_targets/zuo_civ_data/unconstrained2/`. (Original results before the starting value correction and rej_abs_line was made are under unconstrained/)
